# MSLG-SPA 2026 - Bidirectional Gloss <-> Spanish Translation

**Task:** IberLEF 2026 MSLG-SPA shared task
**Model:** mBART-large-50 + LoRA
**Metrics (official):** BLEU + TER + chrF
**System output deadline:** 2026-04-30

---
## Before you run
1. `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`
2. Google Drive must contain:
   ```
   MyDrive/ML_projects/mslg-spa-2026/data/raw/
   |-- MSLG_SPA_train.txt          (training set, required)
   |-- external_spanish.txt        (optional, for back-translation)
   |-- test_mslg2spa.tsv           (required before running predict)
   `-- test_spa2mslg.tsv           (required before running predict)
   ```
3. Run all cells top-to-bottom. Sections 1-4 are idempotent and safe to re-run.
   Section 4 (checkpoint restore) is a no-op on first run and auto-resumes on subsequent runs.


## 1 - Environment setup


In [ ]:
# 1.1 - GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
    )

device   = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
print(f"torch {torch.__version__}  |  CUDA {torch.version.cuda}")


In [ ]:
# 1.2 - Mount Drive + configure paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ============================================================
#  CONFIGURE THESE PATHS - edit only here
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/ML_projects/mslg-spa-2026")
DRIVE_DATA = DRIVE_BASE / "data/raw"
DRIVE_CKPT = DRIVE_BASE / "checkpoints"
DRIVE_SUB  = DRIVE_BASE / "submissions"
# ============================================================

PROJECT_ROOT = Path("/content/mslg-spa-2026")
LOCAL_DATA   = Path("/content/data_local")

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_SUB]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive base : {DRIVE_BASE}")
print(f"Drive data : {DRIVE_DATA}")
print(f"Drive ckpt : {DRIVE_CKPT}")
print(f"Drive sub  : {DRIVE_SUB}")


In [ ]:
# 1.3 - Install packages (transformers/peft stack)
# Colab has torch, numpy, pandas, sklearn, pyyaml already.
!pip install -q transformers==4.46.0 peft==0.13.2 sentencepiece==0.2.0 \
    sacrebleu==2.4.3 evaluate==0.4.3 nltk==3.9.1

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
print("Packages installed.")


## 2 - Project source files

Source files are written inline via `%%writefile` from the current repo state.
To update the notebook after editing any source file, re-run `scripts/build_colab_notebook.py`.


In [ ]:
# 2.0 - Directory structure
from pathlib import Path

PROJECT_ROOT = Path("/content/mslg-spa-2026")
for d in [
    "src/data",
    "src/models",
    "src/evaluation",
    "src/training",
    "scripts",
    "configs",
    "data/raw",
    "data/processed",
    "checkpoints",
    "outputs",
]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

for pkg in ["src", "src/data", "src/models", "src/evaluation", "src/training", "scripts"]:
    init = PROJECT_ROOT / pkg / "__init__.py"
    if not init.exists():
        init.write_text("")

print(f"Directory structure created at {PROJECT_ROOT}")


In [ ]:
%%writefile /content/mslg-spa-2026/src/data/dataset.py
# src/data/dataset.py
"""
Dataset loading and preprocessing for MSLG-SPA 2026.

The dataset consists of aligned pairs:
  - MSLG: Mexican Sign Language gloss sequences (e.g., "TÚ LLEGAR TARDE POR QUÉ")
  - SPA:  Spanish sentences (e.g., "¿Por qué llegaste tarde?")

Expected TSV format: two columns, no header.
  Column 0: MSLG gloss sequence
  Column 1: Spanish sentence
"""

import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizer


def load_pairs(filepath: str | Path) -> pd.DataFrame:
    """
    Load a TSV file of gloss/Spanish pairs into a DataFrame.

    Handles both formats:
      - Train:  ID \\t MSLG \\t SPA  (3 columns, header row)
      - Test:   ID \\t MSLG         (2 columns, header row)
               ID \\t SPA          (2 columns, header row)

    Returns a DataFrame with lowercase column names, ID column dropped.
    Train files yield columns ['mslg', 'spa']; test files yield one of the two.

    The source file is UTF-8 encoded (contains Spanish accents É/Á/Ñ/etc).
    Pass encoding='utf-8' explicitly because on Windows pandas defaults to
    the system code page (cp1252) which can misread accented characters.
    """
    df = pd.read_csv(filepath, sep="\t", header=0, encoding="utf-8")
    df.columns = [c.lower() for c in df.columns]
    if "id" in df.columns:
        df = df.drop(columns=["id"])

    df = df.dropna()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    # Remove rows where any text column is empty
    for col in df.select_dtypes(include="object").columns:
        df = df[df[col] != ""]
    df = df.reset_index(drop=True)

    return df


def print_stats(df: pd.DataFrame, name: str = "Dataset") -> None:
    """
    Print basic corpus statistics.
    Useful to run before training to understand the data.
    """
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"  Pairs:             {len(df)}")
    print(f"  Avg MSLG tokens:   {df['mslg'].str.split().str.len().mean():.1f}")
    print(f"  Avg SPA tokens:    {df['spa'].str.split().str.len().mean():.1f}")
    print(f"  Max MSLG tokens:   {df['mslg'].str.split().str.len().max()}")
    print(f"  Max SPA tokens:    {df['spa'].str.split().str.len().max()}")
    print(f"  Unique MSLG types: {len(set(' '.join(df['mslg']).split()))}")
    print(f"  Unique SPA types:  {len(set(' '.join(df['spa']).split()))}")


class TranslationDataset(Dataset):
    """
    PyTorch Dataset for sequence-to-sequence translation.

    Handles both subtask directions:
      - mslg2spa: source=mslg, target=spa
      - spa2mslg: source=spa, target=mslg

    Args:
        data:          DataFrame with columns ['mslg', 'spa'].
        tokenizer:     HuggingFace tokenizer.
        subtask:       'mslg2spa' or 'spa2mslg'.
        max_src_len:   Max token length for source sequences.
        max_tgt_len:   Max token length for target sequences.
        preprocess_fn: Optional callable applied to source strings only.
                       Used for MSL gloss normalization experiments.
    """

    def __init__(
        self,
        data: pd.DataFrame,
        tokenizer: PreTrainedTokenizer,
        subtask: str,
        max_src_len: int = 128,
        max_tgt_len: int = 128,
        preprocess_fn=None,
    ) -> None:

        assert subtask in ("mslg2spa", "spa2mslg"), \
            "subtask must be 'mslg2spa' or 'spa2mslg'"

        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        # Assign source/target based on subtask direction
        if subtask == "mslg2spa":
            sources = data["mslg"].tolist()
            self.targets = data["spa"].tolist()
        else:
            sources = data["spa"].tolist()
            self.targets = data["mslg"].tolist()

        # Apply preprocessing to source side only (targets are never preprocessed)
        if preprocess_fn is not None:
            self.sources = [preprocess_fn(s) for s in sources]
        else:
            self.sources = sources

    def __len__(self) -> int:
        return len(self.sources)

    def __getitem__(self, idx: int) -> dict:
        source = self.sources[idx]
        target = self.targets[idx]

        # Tokenize source
        model_inputs = self.tokenizer(
            source,
            max_length=self.max_src_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        # Tokenize target
        labels = self.tokenizer(
            text_target=target,
            max_length=self.max_tgt_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        # Replace padding token id with -100 so it is ignored in the loss
        label_ids = labels["input_ids"].squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels":         label_ids,
        }

In [ ]:
%%writefile /content/mslg-spa-2026/src/data/preprocessing.py
# src/data/preprocessing.py
"""
MSL gloss preprocessing for MSLG-SPA 2026.

Handles two types of transformations:
  1. Normalization: removes/simplifies rare MSL annotations (dm-, +, #)
  2. Hyphen special token: replaces compound marker - with [HYPHEN]
"""

import re
from transformers import PreTrainedTokenizer


HYPHEN_TOKEN = "[HYPHEN]"


def normalize_msl_glosses(text: str) -> str:
    """Remove rare MSL annotations that appear too infrequently to learn.

    Rules:
      - dm-WORD  -> WORD        (fingerspelling marker, 47 occurrences)
      - WORD+WORD -> WORD WORD  (compound sign, 22 occurrences)
      - #WORD    -> WORD        (number sign, 5 occurrences)
      - Hyphens kept intact (138 occurrences — handled separately)

    Args:
        text: Raw MSL gloss string.

    Returns:
        Normalized gloss string.
    """
    text = re.sub(r'dm-', '', text)
    text = re.sub(r'\+', ' ', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r' +', ' ', text).strip()
    return text


def apply_hyphen_token(text: str) -> str:
    """Replace compound marker hyphen with [HYPHEN] special token.

    In MSL glosses, hyphens always mark compound signs (LICENCIA-DE-CONDUCIR).
    Replacing with a dedicated token prevents BPE from fragmenting the
    compound marker inconsistently across subword units.

    Args:
        text: Gloss string (typically after normalize_msl_glosses).

    Returns:
        Gloss string with - replaced by [HYPHEN].
    """
    return re.sub(r' +', ' ', text.replace('-', f' {HYPHEN_TOKEN} ')).strip()


def preprocess_gloss(text: str, use_hyphen_token: bool = True) -> str:
    """Apply full MSL preprocessing pipeline.

    Args:
        text:              Raw MSL gloss string.
        use_hyphen_token:  If True, replace - with [HYPHEN] special token.

    Returns:
        Preprocessed gloss string.
    """
    text = normalize_msl_glosses(text)
    if use_hyphen_token:
        text = apply_hyphen_token(text)
    return text


def add_hyphen_special_token(
    tokenizer: PreTrainedTokenizer,
    model,
) -> None:
    """Add [HYPHEN] as a special token and resize model embeddings.

    Must be called before training when use_hyphen_token=True.
    The tokenizer and model must both be saved after this call so the
    new vocabulary is persisted.

    Args:
        tokenizer: HuggingFace tokenizer to modify in-place.
        model:     Seq2seq model whose embeddings will be resized.
    """
    if HYPHEN_TOKEN not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({'additional_special_tokens': [HYPHEN_TOKEN]})
        model.resize_token_embeddings(len(tokenizer))


In [ ]:
%%writefile /content/mslg-spa-2026/src/models/seq2seq.py
# src/models/seq2seq.py
"""
Model loading utilities for MSLG-SPA 2026.

We use Helsinki-NLP/opus-mt-es-ROMANCE as our baseline model.
LoRA is applied to reduce overfitting on the small training set (489 pairs).
"""

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


DEFAULT_LORA_TARGET_MODULES = ["q_proj", "v_proj"]


def load_model_and_tokenizer(
    model_name: str,
    use_lora: bool = True,
    lora_r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.1,
    lora_target_modules: list[str] | None = None,
):
    """
    Load a seq2seq model and tokenizer, optionally wrapping with LoRA.

    Args:
        model_name:          HuggingFace model identifier.
        use_lora:            Whether to apply LoRA (recommended for this task).
        lora_r:              LoRA rank — higher means more capacity but more parameters.
        lora_alpha:          LoRA scaling factor (typically 2x r).
        lora_dropout:        Dropout applied to LoRA layers.
        lora_target_modules: List of attention projection names to adapt with LoRA.
                             Defaults to ["q_proj", "v_proj"]. For mBART, valid
                             attention names are q_proj/k_proj/v_proj/out_proj;
                             FFN names are fc1/fc2.

    Returns:
        (model, tokenizer) tuple ready for training or inference.
    """
    # Load tokenizer and model from HuggingFace Hub
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # mBART-50 requires src_lang/tgt_lang for text_target tokenization
    if hasattr(tokenizer, "src_lang") and tokenizer.src_lang is None:
        tokenizer.src_lang = "es_XX"
    if hasattr(tokenizer, "tgt_lang") and tokenizer.tgt_lang is None:
        tokenizer.tgt_lang = "es_XX"
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    if use_lora:
        from peft import LoraConfig, TaskType, get_peft_model

        target_modules = (
            list(lora_target_modules)
            if lora_target_modules is not None
            else list(DEFAULT_LORA_TARGET_MODULES)
        )
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=target_modules,
            bias="none",
        )
        print(f"  [LoRA] target_modules = {target_modules}")

        # Wrap the model with LoRA — freezes base weights,
        # adds small trainable matrices on top
        model = get_peft_model(model, lora_config)

        # Print how many parameters are actually trainable
        model.print_trainable_parameters()

    return model, tokenizer


def count_parameters(model) -> dict[str, int]:
    """
    Return total and trainable parameter counts.
    Useful to verify LoRA is working correctly.

    Returns:
        Dict with keys 'total' and 'trainable'.
    """
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}

In [ ]:
%%writefile /content/mslg-spa-2026/src/evaluation/metrics.py
# src/evaluation/metrics.py
"""
Metric computation for MSLG-SPA 2026.

Official metrics (confirmed from official evaluation protocol):
  MSLG2SPA: BLEU, METEOR, chrF, COMET
  SPA2MSLG: BLEU, METEOR, chrF  (COMET not applied — gloss is not natural language)

Ranking: z-score normalization across submitted systems, then arithmetic mean.
TER is NOT part of the official ranking — kept only as an optional diagnostic.
"""

import evaluate
import numpy as np


def compute_bleu(predictions: list[str], references: list[str]) -> float:
    """
    Corpus-level BLEU score.

    Args:
        predictions: List of system output strings.
        references:  List of reference strings.

    Returns:
        BLEU score in [0, 100].
    """
    metric = evaluate.load("sacrebleu")
    result = metric.compute(
        predictions=predictions,
        references=[[r] for r in references],  # sacrebleu expects list of lists
    )
    return result["score"]


def compute_chrf(predictions: list[str], references: list[str]) -> float:
    """
    Corpus-level chrF score (character n-gram F-score).
    More robust than BLEU on short sequences and small datasets.

    Returns:
        chrF score in [0, 100].
    """
    metric = evaluate.load("chrf")
    result = metric.compute(
        predictions=predictions, references=[[r] for r in references]
    )
    return result["score"]


def compute_meteor(predictions: list[str], references: list[str]) -> float:
    """
    Corpus-level METEOR score. Official metric for both subtasks.

    Returns:
        METEOR score in [0, 1].
    """
    metric = evaluate.load("meteor")
    result = metric.compute(predictions=predictions, references=references)
    return result["meteor"]


def compute_comet(
    sources: list[str],
    predictions: list[str],
    references: list[str],
) -> float:
    """
    COMET score — official metric for MSLG2SPA only.
    Measures adequacy and fluency using a pretrained neural model.
    Requires the `unbabel-comet` package and GPU for reasonable speed.

    Returns:
        COMET system-level score in roughly [-1, 1], or NaN if not installed.
    """
    try:
        from comet import download_model, load_from_checkpoint

        model_path = download_model("Unbabel/wmt22-comet-da")
        comet_model = load_from_checkpoint(model_path)
        data = [
            {"src": s, "mt": p, "ref": r}
            for s, p, r in zip(sources, predictions, references)
        ]
        output = comet_model.predict(data, batch_size=8, gpus=0)
        return output["system_score"]
    except ImportError:
        print("[WARNING] unbabel-comet not installed. Skipping COMET.")
        return float("nan")


def compute_ter(predictions: list[str], references: list[str]) -> float:
    """
    Corpus-level TER (Translation Edit Rate) — diagnostic only, NOT official.

    Returns:
        TER score in [0, 100] — lower is better.
    """
    metric = evaluate.load("ter")
    result = metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )
    return result["score"]


def evaluate_subtask(
    sources: list[str],
    predictions: list[str],
    references: list[str],
    subtask: str,
    include_comet: bool = False,
    include_ter: bool = False,
) -> dict[str, float]:
    """
    Compute official metrics for a given subtask and print results.

    Official IberLEF 2026 MSLG-SPA metrics:
      - MSLG2SPA: BLEU + METEOR + chrF + COMET (COMET opt-in via include_comet)
      - SPA2MSLG: BLEU + METEOR + chrF

    Args:
        sources:       Source sentences (required for COMET).
        predictions:   System outputs.
        references:    Gold references.
        subtask:       'mslg2spa' or 'spa2mslg'.
        include_comet: Compute COMET for mslg2spa (slow — needs unbabel-comet + GPU).
        include_ter:   Compute TER as an extra diagnostic (not in official ranking).

    Returns:
        Dictionary with metric names as keys and scores as values.
    """
    assert subtask in ("mslg2spa", "spa2mslg")

    results: dict[str, float] = {}
    results["bleu"] = compute_bleu(predictions, references)
    results["meteor"] = compute_meteor(predictions, references)
    results["chrf"] = compute_chrf(predictions, references)

    if subtask == "mslg2spa" and include_comet:
        results["comet"] = compute_comet(sources, predictions, references)

    if include_ter:
        results["ter"] = compute_ter(predictions, references)

    # Print results table
    print(f"\n{'=' * 40}")
    print(f"  Results — {subtask.upper()}")
    print(f"{'=' * 40}")
    official = {"bleu", "meteor", "chrf", "comet"}
    for k, v in results.items():
        note = "  (diagnostic)" if k not in official else ""
        print(f"  {k.upper():<10}: {v:.4f}{note}")

    return results


def compute_global_score(
    scores_per_system: list[dict[str, float]],
    subtask: str,
) -> list[float]:
    """
    Replicate the official IberLEF 2026 Global Score for internal ablations.

    Official method: z-score normalize each metric across systems, then take
    the arithmetic mean of the normalized scores.

    Metrics used:
      - MSLG2SPA: bleu, meteor, chrf, comet (if present)
      - SPA2MSLG: bleu, meteor, chrf

    Args:
        scores_per_system: List of metric dicts, one per system.
        subtask:           'mslg2spa' or 'spa2mslg'.

    Returns:
        List of Global Scores, one per system (higher is better).
    """
    assert subtask in ("mslg2spa", "spa2mslg")

    base_metrics = ["bleu", "meteor", "chrf"]
    if subtask == "mslg2spa" and all("comet" in s for s in scores_per_system):
        base_metrics = ["bleu", "meteor", "chrf", "comet"]

    matrix = np.array(
        [[s[m] for m in base_metrics] for s in scores_per_system],
        dtype=float,
    )

    means = matrix.mean(axis=0)
    stds = matrix.std(axis=0)
    stds[stds == 0] = 1.0

    normalized = (matrix - means) / stds
    global_scores = normalized.mean(axis=1).tolist()

    return global_scores


In [ ]:
%%writefile /content/mslg-spa-2026/scripts/train.py
# scripts/train.py
"""
Training entry point for MSLG-SPA 2026.

Usage:
    python scripts/train.py --config configs/baseline.yaml --subtask mslg2spa
    python scripts/train.py --config configs/baseline.yaml --subtask spa2mslg
"""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import argparse
import yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    TrainerCallback,
)
import evaluate

from src.data.dataset import load_pairs, print_stats, TranslationDataset
from src.models.seq2seq import load_model_and_tokenizer


class BestModelCallback(TrainerCallback):
    """Prints a message on new best and immediately backs up the checkpoint to Drive.

    Uses a _pending_backup flag set in on_evaluate (where we detect the new best)
    and consumed in on_save (where the checkpoint is guaranteed to be on disk).
    This avoids relying on state.best_model_checkpoint timing inside the Trainer.
    """

    def __init__(
        self, metric_name: str, drive_ckpt_dir: str | None = None, subtask: str = ""
    ) -> None:
        self.metric_name = metric_name
        self.drive_ckpt_dir = Path(drive_ckpt_dir) if drive_ckpt_dir else None
        self.subtask = subtask
        self.best_score: float = -float("inf")
        self._pending_backup: bool = False

    def on_evaluate(self, args, state, control, metrics=None, **kwargs) -> None:
        if metrics is None:
            return
        score = metrics.get(f"eval_{self.metric_name}")
        if score is not None and score > self.best_score:
            self.best_score = score
            self._pending_backup = True
            print(
                f"\n*** NEW BEST — epoch {metrics.get('epoch', '?'):.1f} | "
                f"{self.metric_name} = {score:.4f} *** checkpoint saved\n"
            )

    def on_save(self, args, state, control, **kwargs) -> None:
        """Triggered right after the Trainer writes a checkpoint to disk."""
        if not self._pending_backup:
            return
        if self.drive_ckpt_dir is None:
            print("  [Drive backup] SKIPPED — --drive_ckpt_dir not set")
            return
        self._pending_backup = False
        import shutil

        src = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not src.exists():
            print(f"  [Drive backup] SKIPPED — checkpoint not found at {src.resolve()}")
            return
        # Subtask-scoped Drive path: DRIVE_CKPT/mslg2spa/checkpoint-N
        dst = self.drive_ckpt_dir / self.subtask / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  [Drive backup] {src.parent.name}/{src.name} → {dst}")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, help="Path to YAML config file")
    parser.add_argument("--subtask", required=True, choices=["mslg2spa", "spa2mslg"])
    parser.add_argument(
        "--drive_ckpt_dir", default=None, help="Drive checkpoint dir for live backup"
    )
    return parser.parse_args()


def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def make_compute_metrics(tokenizer, subtask):
    """
    Returns a compute_metrics function for the HuggingFace Trainer.
    Trainer calls this function at the end of each evaluation epoch.
    """
    chrf_metric = evaluate.load("chrf")
    bleu_metric = evaluate.load("sacrebleu")

    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        # Clip predictions to valid token range before decoding
        preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

        # Replace -100 (padding) with pad_token_id before decoding
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        # Decode token ids back to strings
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # Strip whitespace
        decoded_preds = [p.strip() for p in decoded_preds]
        decoded_labels = [l.strip() for l in decoded_labels]

        # Compute metrics
        chrf = chrf_metric.compute(
            predictions=decoded_preds, references=[[r] for r in decoded_labels]
        )
        bleu = bleu_metric.compute(
            predictions=decoded_preds, references=[[r] for r in decoded_labels]
        )

        return {
            "chrf": chrf["score"],
            "bleu": bleu["score"],
        }

    return compute_metrics


def main():
    args = parse_args()
    config = load_config(args.config)

    # ------------------------------------------------------------------ #
    # 1. Load data
    # ------------------------------------------------------------------ #
    # If real_train_file is set, carve val from real data only to avoid
    # synthetic pairs leaking into validation (data integrity: val must be
    # 100% real, regardless of whether train_file contains augmented data).
    real_file = config["data"].get("real_train_file") or config["data"]["train_file"]
    real_df = load_pairs(real_file)
    print_stats(real_df, name="Real training data")

    train_df, val_df = train_test_split(
        real_df,
        test_size=config["data"]["val_split"],
        random_state=config["training"]["seed"],
    )

    # If an augmented file is provided, append synthetic pairs to train only
    aug_file = config["data"].get("train_file")
    if aug_file and aug_file != real_file:
        aug_df = load_pairs(aug_file)
        synthetic_df = aug_df[~aug_df.index.isin(real_df.index)].copy()
        # real_df may not share index with aug_df; deduplicate by content instead
        real_set = set(zip(real_df.iloc[:, 0], real_df.iloc[:, 1]))
        synthetic_df = aug_df[
            ~aug_df.apply(lambda r: (r.iloc[0], r.iloc[1]) in real_set, axis=1)
        ]
        train_df = pd.concat([train_df, synthetic_df], ignore_index=True)
        print(f"  Synthetic pairs appended to train: {len(synthetic_df)}")

    print(f"\n  Train pairs (total): {len(train_df)}")
    print(f"  Val pairs (real only): {len(val_df)}")

    # ------------------------------------------------------------------ #
    # 2. Load model and tokenizer
    # ------------------------------------------------------------------ #
    model, tokenizer = load_model_and_tokenizer(
        model_name=config["model"]["name"],
        use_lora=config["lora"]["enabled"],
        lora_r=config["lora"]["r"],
        lora_alpha=config["lora"]["lora_alpha"],
        lora_dropout=config["lora"]["lora_dropout"],
        lora_target_modules=config["lora"].get("target_modules"),
    )

    # ------------------------------------------------------------------ #
    # 3. Preprocessing (optional)
    # ------------------------------------------------------------------ #
    preprocess_fn = None
    prep_cfg = config.get("preprocessing", {})
    if prep_cfg.get("enabled", False):
        from src.data.preprocessing import preprocess_gloss, add_hyphen_special_token

        use_hyphen = prep_cfg.get("hyphen_special_token", False)
        preprocess_fn = lambda text: preprocess_gloss(text, use_hyphen_token=use_hyphen)
        if use_hyphen:
            add_hyphen_special_token(tokenizer, model)
            print("  [preprocessing] Added [HYPHEN] special token, embeddings resized")
        print(f"  [preprocessing] enabled  |  hyphen_token={use_hyphen}")

    # ------------------------------------------------------------------ #
    # 4. Build datasets
    # ------------------------------------------------------------------ #
    train_dataset = TranslationDataset(
        data=train_df,
        tokenizer=tokenizer,
        subtask=args.subtask,
        max_src_len=config["model"]["max_source_length"],
        max_tgt_len=config["model"]["max_target_length"],
        preprocess_fn=preprocess_fn,
    )
    val_dataset = TranslationDataset(
        data=val_df,
        tokenizer=tokenizer,
        subtask=args.subtask,
        max_src_len=config["model"]["max_source_length"],
        max_tgt_len=config["model"]["max_target_length"],
        preprocess_fn=preprocess_fn,
    )

    # ------------------------------------------------------------------ #
    # 5. Training arguments
    # ------------------------------------------------------------------ #
    # Append subtask name so mslg2spa and spa2mslg don't overwrite each other
    subtask_output_dir = str(Path(config["training"]["output_dir"]) / args.subtask)

    training_args = Seq2SeqTrainingArguments(
        output_dir=subtask_output_dir,
        num_train_epochs=config["training"]["num_train_epochs"],
        per_device_train_batch_size=config["training"]["per_device_train_batch_size"],
        per_device_eval_batch_size=config["training"]["per_device_eval_batch_size"],
        learning_rate=config["training"]["learning_rate"],
        warmup_steps=config["training"]["warmup_steps"],
        weight_decay=config["training"]["weight_decay"],
        eval_strategy=config["training"]["eval_strategy"],
        save_strategy=config["training"]["save_strategy"],
        load_best_model_at_end=config["training"]["load_best_model_at_end"],
        save_total_limit=config["training"].get("save_total_limit", 1),
        metric_for_best_model=config["training"]["metric_for_best_model"],
        greater_is_better=config["training"]["greater_is_better"],
        predict_with_generate=True,  # needed for seq2seq evaluation
        fp16=config["training"]["fp16"],
        max_grad_norm=config["training"].get("max_grad_norm", 1.0),
        seed=config["training"]["seed"],
        label_smoothing_factor=config["training"].get("label_smoothing_factor", 0.0),
        # Opt-in: align eval-time decoding with final beam search. Off by default
        # so existing baseline.yaml runs are bit-for-bit identical.
        generation_num_beams=(
            config["generation"].get("num_beams")
            if config["training"].get("eval_with_beam_search", False)
            else None
        ),
        generation_max_length=(
            config["generation"].get("max_new_tokens")
            if config["training"].get("eval_with_beam_search", False)
            else None
        ),
        report_to=config["logging"]["report_to"],
        logging_steps=config["logging"]["logging_steps"],
        run_name=config["logging"].get("run_name"),
    )

    # ------------------------------------------------------------------ #
    # 6. Trainer
    # ------------------------------------------------------------------ #
    callbacks = []
    patience = config["training"].get("early_stopping_patience")
    if patience:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=patience))
    callbacks.append(
        BestModelCallback(
            config["training"]["metric_for_best_model"],
            args.drive_ckpt_dir,
            args.subtask,
        )
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
        compute_metrics=make_compute_metrics(tokenizer, args.subtask),
        callbacks=callbacks if callbacks else None,
    )

    # ------------------------------------------------------------------ #
    # 7. Train
    # ------------------------------------------------------------------ #
    print(f"\nStarting training — subtask: {args.subtask}")
    trainer.train()

    # Save final model
    output_dir = Path(config["training"]["output_dir"]) / args.subtask
    trainer.save_model(output_dir / "final")
    tokenizer.save_pretrained(output_dir / "final")
    print(f"\nModel saved to {output_dir / 'final'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mslg-spa-2026/scripts/run_evaluate.py
# scripts/evaluate.py
"""
Evaluation entry point for MSLG-SPA 2026.

Usage:
    python scripts/evaluate.py --config configs/baseline.yaml --subtask mslg2spa
    python scripts/evaluate.py --config configs/baseline.yaml --subtask spa2mslg
"""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import argparse
import yaml
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel
import torch

from src.data.dataset import load_pairs
from src.evaluation.metrics import evaluate_subtask


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--subtask", required=True, choices=["mslg2spa", "spa2mslg"])
    return parser.parse_args()


def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def load_trained_model(checkpoint_dir: str):
    from transformers import MBart50Tokenizer

    checkpoint_dir = Path(checkpoint_dir)

    # Load tokenizer from base model — local checkpoint may lack tokenizer files
    tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50")
    # mBART-50 requires src_lang/tgt_lang for text_target tokenization
    tokenizer.src_lang = "es_XX"
    tokenizer.tgt_lang = "es_XX"

    if (checkpoint_dir / "adapter_config.json").exists():
        import json

        adapter_config = json.load(open(checkpoint_dir / "adapter_config.json"))
        base_model_name = adapter_config["base_model_name_or_path"]

        base_model = AutoModelForSeq2SeqLM.from_pretrained(
            base_model_name, local_files_only=False
        )
        model = PeftModel.from_pretrained(
            base_model, str(checkpoint_dir), local_files_only=False
        )
        model = model.merge_and_unload()
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            str(checkpoint_dir), local_files_only=False
        )
    model.eval()
    return model, tokenizer


def generate_translations(
    model,
    tokenizer,
    sources: list[str],
    subtask: str = "mslg2spa",
    max_src_len: int = 128,
    max_new_tokens: int = 128,
    num_beams: int = 4,
    length_penalty: float = 1.0,
    no_repeat_ngram_size: int = 0,
) -> list[str]:
    """
    Generate translations for a list of source sentences.

    Args:
        model:                Trained seq2seq model.
        tokenizer:            Corresponding tokenizer.
        sources:              List of source sentences to translate.
        subtask:              'mslg2spa' or 'spa2mslg' — determines target language token.
        max_src_len:          Max tokenization length for sources.
        max_new_tokens:       Max tokens to generate per translation.
        num_beams:             Beam search width.
        length_penalty:       Exponential penalty to beam scores; >1 favors longer
                              outputs, <1 favors shorter. 1.0 = neutral (HF default).
        no_repeat_ngram_size: If >0, block repetition of n-grams of this size in
                              generation. Typical: 0 (off) or 3. Useful for short
                              glosses where mBART can loop.

    Returns:
        List of translated strings.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # mBART requires forced_bos_token_id to set the target language
    # MSLG2SPA: target is Spanish; SPA2MSLG: target is Spanish (MSL has no mBART code)
    forced_bos_token_id = tokenizer.lang_code_to_id.get("es_XX")

    translations = []

    # Process one sentence at a time to keep memory usage low
    for source in sources:
        inputs = tokenizer(
            source,
            return_tensors="pt",
            max_length=max_src_len,
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                early_stopping=True,
                forced_bos_token_id=forced_bos_token_id,
                length_penalty=length_penalty,
                no_repeat_ngram_size=no_repeat_ngram_size,
            )

        translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        translations.append(translation)

    return translations


def main():
    args = parse_args()
    config = load_config(args.config)

    # ------------------------------------------------------------------ #
    # 1. Load test data
    # ------------------------------------------------------------------ #
    if args.subtask == "mslg2spa":
        test_file = config["data"]["test_mslg2spa"]
        src_col, tgt_col = "mslg", "spa"
    else:
        test_file = config["data"]["test_spa2mslg"]
        src_col, tgt_col = "spa", "mslg"

    df = load_pairs(test_file)
    sources = df[src_col].tolist()
    references = df[tgt_col].tolist()

    print(f"Loaded {len(df)} test pairs for {args.subtask}")

    # ------------------------------------------------------------------ #
    # 2. Load trained model
    # ------------------------------------------------------------------ #
    checkpoint_dir = Path(config["training"]["output_dir"]) / args.subtask / "final"
    model, tokenizer = load_trained_model(str(checkpoint_dir))
    print(f"Loaded model from {checkpoint_dir}")

    # ------------------------------------------------------------------ #
    # 3. Generate translations
    # ------------------------------------------------------------------ #
    print("Generating translations...")
    predictions = generate_translations(
        model=model,
        tokenizer=tokenizer,
        sources=sources,
        subtask=args.subtask,
        max_src_len=config["model"]["max_source_length"],
        max_new_tokens=config["generation"]["max_new_tokens"],
        num_beams=config["generation"]["num_beams"],
        length_penalty=config["generation"].get("length_penalty", 1.0),
        no_repeat_ngram_size=config["generation"].get("no_repeat_ngram_size", 0),
    )

    # ------------------------------------------------------------------ #
    # 4. Evaluate
    # ------------------------------------------------------------------ #
    evaluate_subtask(
        sources=sources,
        predictions=predictions,
        references=references,
        subtask=args.subtask,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mslg-spa-2026/scripts/predict.py
# scripts/predict.py
"""
Generate official submission files for MSLG-SPA 2026.

Usage:
    python scripts/predict.py --config configs/baseline.yaml --subtask mslg2spa --team YourTeamName --solution baseline
    python scripts/predict.py --config configs/baseline.yaml --subtask spa2mslg --team YourTeamName --solution baseline

Output files follow the official naming convention:
    YourTeamName_baseline_MSLG2SPA.txt
    YourTeamName_baseline_SPA2MSLG.txt
"""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import argparse
import yaml

from src.data.dataset import load_pairs
from scripts.run_evaluate import load_trained_model, generate_translations


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--subtask", required=True, choices=["mslg2spa", "spa2mslg"])
    parser.add_argument("--team", required=True, help="Your team name")
    parser.add_argument(
        "--solution", required=True, help="Solution name (e.g. baseline, lora_r32)"
    )
    return parser.parse_args()


def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def write_submission(
    predictions: list[str],
    output_path: Path,
    ids: list[int] | None = None,
) -> None:
    """Write predictions in the official IberLEF submission format.

    Official format (no ID):
        "SystemOutput"\\n

    Optional format (with instance ID for alignment verification):
        "InstanceIdentifier"\\t"SystemOutput"\\n

    Quotation marks are mandatory. Linux newlines required.
    """
    with open(output_path, "w", encoding="utf-8", newline="\n") as f:
        if ids is not None:
            for id_, pred in zip(ids, predictions):
                f.write(f'"{id_}"\t"{pred}"\n')
        else:
            for pred in predictions:
                f.write(f'"{pred}"\n')

    print(f"Submission saved to {output_path}")
    print(f"Lines written: {len(predictions)}")


def main():
    args = parse_args()
    config = load_config(args.config)

    # ------------------------------------------------------------------ #
    # 1. Load test data (no references — this is the real test set)
    # ------------------------------------------------------------------ #
    if args.subtask == "mslg2spa":
        test_file = config["data"]["test_mslg2spa"]
        src_col = "mslg"
    else:
        test_file = config["data"]["test_spa2mslg"]
        src_col = "spa"

    import pandas as pd

    raw_df = pd.read_csv(test_file, sep="\t", header=0, encoding="utf-8")
    ids = raw_df["ID"].tolist()
    df = load_pairs(test_file)
    sources = df[src_col].tolist()

    print(f"Loaded {len(sources)} test instances for {args.subtask}")

    # ------------------------------------------------------------------ #
    # 2. Load trained model
    # ------------------------------------------------------------------ #
    checkpoint_dir = Path(config["training"]["output_dir"]) / args.subtask / "final"
    model, tokenizer = load_trained_model(str(checkpoint_dir))

    # ------------------------------------------------------------------ #
    # 3. Generate translations
    # ------------------------------------------------------------------ #
    print("Generating translations...")
    predictions = generate_translations(
        model=model,
        tokenizer=tokenizer,
        sources=sources,
        subtask=args.subtask,
        max_src_len=config["model"]["max_source_length"],
        max_new_tokens=config["generation"]["max_new_tokens"],
        num_beams=config["generation"]["num_beams"],
        length_penalty=config["generation"].get("length_penalty", 1.0),
        no_repeat_ngram_size=config["generation"].get("no_repeat_ngram_size", 0),
    )

    # ------------------------------------------------------------------ #
    # 4. Write submission file
    # ------------------------------------------------------------------ #
    # Official naming convention: TeamName_SolutionName_SUBTASK.txt
    filename = f"{args.team}_{args.solution}_{args.subtask.upper()}.txt"
    output_path = Path("outputs") / filename
    output_path.parent.mkdir(exist_ok=True)

    write_submission(predictions, output_path, ids=ids)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mslg-spa-2026/scripts/ensemble_predict.py
# scripts/ensemble_predict.py
"""
Ensemble prediction for MSLG-SPA 2026.

Loads top-N checkpoints by eval_chrf from trainer_state.json,
generates translations from each, and selects the best translation
per sentence via self-consistency voting (mean chrF against the other N-1).

Usage (submission mode — requires test files):
    python scripts/ensemble_predict.py \\
        --config configs/baseline.yaml \\
        --subtask mslg2spa \\
        --checkpoint_dir checkpoints/mslg2spa \\
        --team YourTeam \\
        --solution ensemble3 \\
        --n_checkpoints 3

Usage (validation mode — compares single-best vs ensemble on val split):
    python scripts/ensemble_predict.py \\
        --config configs/baseline.yaml \\
        --subtask mslg2spa \\
        --checkpoint_dir checkpoints/mslg2spa \\
        --validate \\
        --n_checkpoints 3
"""

import sys
from pathlib import Path

# Ensure project root is on sys.path when running as a script
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import argparse
import json
import yaml

from sklearn.model_selection import train_test_split

from src.data.dataset import load_pairs
from src.evaluation.metrics import compute_chrf
from scripts.run_evaluate import load_trained_model, generate_translations


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, help="Path to YAML config file")
    parser.add_argument("--subtask", required=True, choices=["mslg2spa", "spa2mslg"])
    parser.add_argument(
        "--checkpoint_dir",
        required=True,
        help="Directory containing checkpoint-* subdirectories",
    )
    parser.add_argument(
        "--team", default=None, help="Your team name (required unless --validate)"
    )
    parser.add_argument(
        "--solution",
        default=None,
        help="Solution label, e.g. ensemble3 (required unless --validate)",
    )
    parser.add_argument(
        "--n_checkpoints",
        type=int,
        default=3,
        help="Number of top checkpoints to ensemble (default: 3)",
    )
    parser.add_argument(
        "--validate",
        action="store_true",
        help="Run on val split (recreated from train_file with same seed)"
        " and report chrF for single-best vs ensemble. "
        "No submission file is written.",
    )
    args = parser.parse_args()

    if not args.validate and (args.team is None or args.solution is None):
        parser.error("--team and --solution are required unless --validate is set")

    return args


def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def find_top_checkpoints(checkpoint_dir: Path, n: int) -> list[Path]:
    """Find top-N checkpoints by eval_chrf from trainer_state.json.

    Reads trainer_state.json from each checkpoint-* subdirectory,
    extracts the eval_chrf logged at that checkpoint's step, sorts
    descending, and returns the top-N paths.

    Args:
        checkpoint_dir: Directory containing checkpoint-* subdirectories.
        n:              Number of top checkpoints to return.

    Returns:
        List of checkpoint Paths sorted by eval_chrf descending.

    Raises:
        ValueError: If no checkpoints with trainer_state.json are found.
    """
    scored: list[tuple[float, Path]] = []

    for ckpt in sorted(checkpoint_dir.glob("checkpoint-*")):
        if not ckpt.is_dir():
            continue
        state_file = ckpt / "trainer_state.json"
        if not state_file.exists():
            continue

        state = json.load(open(state_file))
        ckpt_step = int(ckpt.name.split("-")[1])

        # Search log_history for the eval_chrf at this checkpoint's step
        chrf_score = None
        for entry in state.get("log_history", []):
            if entry.get("step") == ckpt_step and "eval_chrf" in entry:
                chrf_score = entry["eval_chrf"]
                break

        # Fallback: if this is the best checkpoint, use best_metric
        if chrf_score is None:
            best_ckpt_path = state.get("best_model_checkpoint", "")
            if best_ckpt_path and Path(best_ckpt_path).name == ckpt.name:
                chrf_score = state.get("best_metric")

        if chrf_score is not None:
            scored.append((chrf_score, ckpt))

    if not scored:
        raise ValueError(
            f"No checkpoints with trainer_state.json and eval_chrf found in "
            f"{checkpoint_dir}. Make sure training completed and checkpoints are saved."
        )

    scored.sort(key=lambda x: x[0], reverse=True)
    top_n = min(n, len(scored))
    print(f"\nTop-{top_n} checkpoints by eval_chrf:")
    for score, ckpt in scored[:top_n]:
        print(f"  {ckpt.name}  chrF={score:.2f}")

    return [ckpt for _, ckpt in scored[:top_n]]


def self_consistency_vote(predictions_per_model: list[list[str]]) -> list[str]:
    """For each sentence, pick the translation with highest mean chrF against others.

    If all N models agree on a translation, that translation is returned directly.
    Otherwise, each candidate is scored by computing its mean chrF against the
    other N-1 candidates, and the highest-scoring candidate is selected.

    Args:
        predictions_per_model: List of N prediction lists, one per checkpoint.
                               Each inner list has one string per sentence.

    Returns:
        List of selected translations, one per sentence.
    """
    n_models = len(predictions_per_model)
    n_sentences = len(predictions_per_model[0])
    best = []

    for i in range(n_sentences):
        candidates = [predictions_per_model[m][i] for m in range(n_models)]

        # Fast path: unanimous agreement
        if len(set(candidates)) == 1:
            best.append(candidates[0])
            continue

        # Score each candidate by mean chrF against the other N-1
        scores = []
        for j, candidate in enumerate(candidates):
            others = [c for k, c in enumerate(candidates) if k != j]
            mean_chrf = compute_chrf([candidate] * len(others), others) / 100.0
            scores.append(mean_chrf)

        best.append(candidates[scores.index(max(scores))])

    return best


def write_submission(predictions: list[str], output_path: Path) -> None:
    """Write predictions in the official IberLEF submission format.

    Each line: "SystemOutput"\\n
    """
    with open(output_path, "w", encoding="utf-8", newline="\n") as f:
        for pred in predictions:
            f.write(f'"{pred}"\n')
    print(f"\nSubmission saved to {output_path}  ({len(predictions)} lines)")


def load_validation_sources_and_refs(
    config: dict, subtask: str
) -> tuple[list[str], list[str]]:
    """Recreate the val split from train_file using the same seed as train.py.

    Mirrors the split logic in ``scripts/train.py`` so that the validation set
    used here is bit-identical to the one used during training.

    Returns:
        (sources, references) lists for the given subtask direction.
    """
    real_file = config["data"].get("real_train_file") or config["data"]["train_file"]
    df = load_pairs(real_file)
    _, val_df = train_test_split(
        df,
        test_size=config["data"]["val_split"],
        random_state=config["training"]["seed"],
    )
    val_df = val_df.reset_index(drop=True)

    if subtask == "mslg2spa":
        src_col, tgt_col = "mslg", "spa"
    else:
        src_col, tgt_col = "spa", "mslg"

    return val_df[src_col].tolist(), val_df[tgt_col].tolist()


def main():
    args = parse_args()
    config = load_config(args.config)

    # ------------------------------------------------------------------ #
    # 1. Load sources (val split in validate mode, test file otherwise)
    # ------------------------------------------------------------------ #
    references: list[str] | None = None

    if args.validate:
        sources, references = load_validation_sources_and_refs(config, args.subtask)
        print(f"[VALIDATE] Loaded {len(sources)} val instances for {args.subtask}")
    else:
        if args.subtask == "mslg2spa":
            test_file = config["data"]["test_mslg2spa"]
            src_col = "mslg"
        else:
            test_file = config["data"]["test_spa2mslg"]
            src_col = "spa"

        df = load_pairs(test_file)
        sources = df[src_col].tolist()
        print(f"Loaded {len(sources)} test instances for {args.subtask}")

    # ------------------------------------------------------------------ #
    # 2. Find top-N checkpoints by eval_chrf
    # ------------------------------------------------------------------ #
    checkpoint_dir = Path(args.checkpoint_dir)
    top_checkpoints = find_top_checkpoints(checkpoint_dir, args.n_checkpoints)

    # ------------------------------------------------------------------ #
    # 3. Generate translations from each checkpoint
    # ------------------------------------------------------------------ #
    all_predictions: list[list[str]] = []
    for ckpt in top_checkpoints:
        print(f"\nLoading {ckpt.name}...")
        model, tokenizer = load_trained_model(str(ckpt))
        preds = generate_translations(
            model=model,
            tokenizer=tokenizer,
            sources=sources,
            subtask=args.subtask,
            max_src_len=config["model"]["max_source_length"],
            max_new_tokens=config["generation"]["max_new_tokens"],
            num_beams=config["generation"]["num_beams"],
            length_penalty=config["generation"].get("length_penalty", 1.0),
            no_repeat_ngram_size=config["generation"].get("no_repeat_ngram_size", 0),
        )
        all_predictions.append(preds)
        del model  # free GPU memory between checkpoints

    # ------------------------------------------------------------------ #
    # 4. Self-consistency vote
    # ------------------------------------------------------------------ #
    print("\nApplying self-consistency vote...")
    final_predictions = self_consistency_vote(all_predictions)

    # ------------------------------------------------------------------ #
    # 5a. Validate mode: report single-best vs ensemble chrF on val split
    # ------------------------------------------------------------------ #
    if args.validate:
        assert references is not None

        print(f"\n{'=' * 56}")
        print(f"  Validation results — {args.subtask.upper()}")
        print(f"{'=' * 56}")

        # Per-checkpoint chrF (top checkpoint is index 0 = single-best baseline)
        per_ckpt_chrf = []
        for ckpt, preds in zip(top_checkpoints, all_predictions):
            score = compute_chrf(preds, references)
            per_ckpt_chrf.append(score)
            print(f"  {ckpt.name:<30}  chrF = {score:.4f}")

        single_best_chrf = per_ckpt_chrf[0]
        ensemble_chrf = compute_chrf(final_predictions, references)
        delta = ensemble_chrf - single_best_chrf

        print(f"\n  {'single-best (top-1)':<30}  chrF = {single_best_chrf:.4f}")
        print(f"  {'ensemble (self-cons vote)':<30}  chrF = {ensemble_chrf:.4f}")
        print(f"  {'delta':<30}         {delta:+.4f}")
        print(f"{'=' * 56}\n")
        return

    # ------------------------------------------------------------------ #
    # 5b. Submission mode: write output file
    # ------------------------------------------------------------------ #
    filename = f"{args.team}_{args.solution}_{args.subtask.upper()}.txt"
    output_path = Path("outputs") / filename
    output_path.parent.mkdir(exist_ok=True)
    write_submission(final_predictions, output_path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mslg-spa-2026/configs/baseline.yaml
# Model configuration
model:
  name: "facebook/mbart-large-50"
  max_source_length: 128
  max_target_length: 128

# LoRA
lora:
  enabled: true
  r: 16
  lora_alpha: 32
  lora_dropout: 0.1
  # mBART attention projections. Default baseline uses q+v only (matches the
  # chrF 52.15 reference run). See configs/strong.yaml for q+k+v+out variant.
  target_modules: ["q_proj", "v_proj"]

# Data paths
data:
  train_file: "data/raw/MSLG_SPA_train.txt"
  test_mslg2spa: "data/raw/MSLG2SPA_test.txt"
  test_spa2mslg: "data/raw/SPA2MSLG_test.txt"
  processed_dir: "data/processed/"
  val_split: 0.15

# Training
training:
  output_dir: "checkpoints/baseline"
  num_train_epochs: 40
  per_device_train_batch_size: 8
  per_device_eval_batch_size: 16
  learning_rate: 5.0e-4
  warmup_steps: 50
  weight_decay: 0.01
  eval_strategy: "epoch"
  save_strategy: "epoch"
  load_best_model_at_end: true
  save_total_limit: 3
  early_stopping_patience: 10
  metric_for_best_model: "chrf"
  greater_is_better: true
  fp16: True
  seed: 42
  # 0.0 = no label smoothing (baseline default). strong.yaml enables 0.1.
  label_smoothing_factor: 0.0
  # If true, eval-time decoding uses beam search with the settings below.
  # Default false preserves the legacy baseline behavior (greedy eval during train).
  eval_with_beam_search: false

# Generation
generation:
  num_beams: 5
  max_new_tokens: 128
  # 1.0 = neutral. <1 shortens, >1 lengthens. HF default.
  length_penalty: 1.0
  # 0 = off. Set to 3 to block trigram repetition (useful for mBART loops).
  no_repeat_ngram_size: 0

# Logging
logging:
  report_to: "none"
  logging_steps: 10

In [ ]:
%%writefile /content/mslg-spa-2026/configs/strong.yaml
# configs/strong.yaml
#
# Aggressive config aligned with NotebookLM literature findings (2026-04-11):
#   - LoRA target_modules: ALL linear (q,k,v,out_proj,fc1,fc2) — LowRA framework
#     shows better stability/performance on low-resource than q,v only (C16).
#   - LoRA r=64, alpha=64 — recommended for datasets <500 pairs to give enough
#     capacity to absorb gloss-text mapping without overfitting (C17).
#   - label_smoothing_factor: 0.1 — effective on mBART fine-tuning (C22).
#     NOTE: literature also cites SALS (Semantically Aware Label Smoothing) as a
#     specialized variant for gloss training — not implemented here, future work.
#   - num_train_epochs: 40 — SPA2MSLG was still improving late at 30.
#   - eval_with_beam_search: true — align val chrF with final decoding.
#   - no_repeat_ngram_size: 3 — blocks trigram loops on short glosses.
#
# Reference thresholds from literature (NotebookLM Section F):
#   - BARTO fine-tuned baseline on Lara-Ortiz corpus: BLEU-4 35.0
#   - mBART on SIGNUM (~600 pairs) gloss→text: BLEU-4 67.60
#   - mBART on PHOENIX-14T gloss→German: BLEU-4 25.58
#   - Lara-Ortiz + ASL data augmentation: 62 → 85 BLEU
# Current local baseline: BLEU ~24 (with back-translation) — significantly below
# the BARTO 35.0 reference. Closing this gap is the primary goal of strong.yaml.
#
# Keep baseline.yaml as the reference 52.15 chrF run. Do not overwrite its
# checkpoints — this config writes to checkpoints/strong.

# Model configuration
model:
  name: "facebook/mbart-large-50"
  max_source_length: 128
  max_target_length: 128

# LoRA — LowRA-style all-linear targets + higher rank for low-resource
lora:
  enabled: true
  r: 64
  lora_alpha: 64
  lora_dropout: 0.1
  target_modules: ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]

# Data paths (unchanged)
data:
  train_file: "data/raw/MSLG_SPA_train.txt"
  test_mslg2spa: "data/raw/MSLG2SPA_test.txt"
  test_spa2mslg: "data/raw/SPA2MSLG_test.txt"
  processed_dir: "data/processed/"
  val_split: 0.15

# Training
# Memory note: LoRA trainable params jump from 2.36M (baseline) to 34.6M (r=64 +
# 6 target modules = 5.36% of 645M). With Adam optimizer states (2x) this is
# ~70M extra params on GPU. T4 16GB may OOM at batch_size=8 — if so, drop to
# 4 and raise gradient_accumulation_steps to 2 (keeps effective batch=8).
training:
  output_dir: "checkpoints/strong"
  num_train_epochs: 40
  per_device_train_batch_size: 8
  per_device_eval_batch_size: 16
  learning_rate: 5.0e-4
  warmup_steps: 50
  weight_decay: 0.01
  eval_strategy: "epoch"
  save_strategy: "epoch"
  load_best_model_at_end: true
  metric_for_best_model: "chrf"
  greater_is_better: true
  fp16: True
  seed: 42
  label_smoothing_factor: 0.1
  eval_with_beam_search: true

# Generation
generation:
  num_beams: 5
  max_new_tokens: 128
  length_penalty: 1.0
  no_repeat_ngram_size: 3

# Logging
logging:
  report_to: "none"
  logging_steps: 10
  run_name: "strong_v1"


In [ ]:
%%writefile /content/mslg-spa-2026/configs/baseline_bt.yaml
# EXP-004 / EXP-004b — baseline config + back-translated data
#
# Changes vs baseline.yaml:
#   train_file:      MSLG_SPA_train.txt → data/processed/augmented_train.tsv
#   real_train_file: added — val split carved from real data only (no BT leakage)
#   output_dir:      checkpoints/baseline → checkpoints/baseline_bt
#
# All other hyperparameters identical to baseline.yaml.
# Run back_translate.py first to generate augmented_train.tsv.

model:
  name: "facebook/mbart-large-50"
  max_source_length: 128
  max_target_length: 128

lora:
  enabled: true
  r: 16
  lora_alpha: 32
  lora_dropout: 0.1
  target_modules: ["q_proj", "v_proj"]

data:
  train_file: "data/processed/augmented_train.tsv"
  real_train_file: "data/raw/MSLG_SPA_train.txt"   # val carved from real data only
  test_mslg2spa: "data/raw/MSLG2SPA_test.txt"
  test_spa2mslg: "data/raw/SPA2MSLG_test.txt"
  processed_dir: "data/processed/"
  val_split: 0.15

training:
  output_dir: "checkpoints/baseline_bt"
  num_train_epochs: 40
  per_device_train_batch_size: 8
  per_device_eval_batch_size: 16
  learning_rate: 5.0e-4
  warmup_steps: 50
  weight_decay: 0.01
  eval_strategy: "epoch"
  save_strategy: "epoch"
  load_best_model_at_end: true
  save_total_limit: 3
  early_stopping_patience: 10
  metric_for_best_model: "chrf"
  greater_is_better: true
  fp16: true
  seed: 42
  label_smoothing_factor: 0.0
  eval_with_beam_search: false

generation:
  num_beams: 5
  max_new_tokens: 128
  length_penalty: 1.0
  no_repeat_ngram_size: 0

logging:
  report_to: "none"
  logging_steps: 10


## 3 - Data setup

Copies training data from Drive into `/content/data_local/` (RAM). Test files are copied if present; otherwise the cell warns and continues.


In [ ]:
# 3.1 - Copy training + test data from Drive to local RAM
import shutil
from pathlib import Path

LOCAL_DATA = Path("/content/data_local")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

def copy_if_exists(name, required=False):
    src = DRIVE_DATA / name
    dst = LOCAL_DATA / name
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  OK      {name}")
        return True
    msg = f"  MISSING {name}"
    if required:
        raise FileNotFoundError(f"Required file not found on Drive: {src}")
    print(msg + "  (optional — skipping)")
    return False

copy_if_exists("MSLG_SPA_train.txt", required=True)
copy_if_exists("external_spanish.txt", required=False)
has_test_m2s = copy_if_exists("test_mslg2spa.tsv", required=False)
has_test_s2m = copy_if_exists("test_spa2mslg.tsv", required=False)

print()
if not (has_test_m2s and has_test_s2m):
    print("WARNING: Test files missing. Training will work, prediction cells will fail.")
else:
    print("All test files present.")


In [ ]:
# 3.2 - Patch config paths (point both baseline.yaml and strong.yaml to /content/data_local)
import os, sys, yaml

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

for cfg_name in ["configs/baseline.yaml", "configs/strong.yaml"]:
    with open(cfg_name) as f:
        cfg = yaml.safe_load(f)
    cfg["data"]["train_file"]    = str(LOCAL_DATA / "MSLG_SPA_train.txt")
    cfg["data"]["test_mslg2spa"] = str(LOCAL_DATA / "test_mslg2spa.tsv")
    cfg["data"]["test_spa2mslg"] = str(LOCAL_DATA / "test_spa2mslg.tsv")
    with open(cfg_name, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    print(f"Patched {cfg_name}")

# baseline_bt.yaml: patch both real_train_file (val) and train_file (augmented)
_bt_cfg_name = "configs/baseline_bt.yaml"
with open(_bt_cfg_name) as f:
    _bt_cfg = yaml.safe_load(f)
_bt_cfg["data"]["real_train_file"] = str(LOCAL_DATA / "MSLG_SPA_train.txt")
_bt_cfg["data"]["train_file"]      = str(LOCAL_DATA / "augmented_train.tsv")
_bt_cfg["data"]["test_mslg2spa"]   = str(LOCAL_DATA / "test_mslg2spa.tsv")
_bt_cfg["data"]["test_spa2mslg"]   = str(LOCAL_DATA / "test_spa2mslg.tsv")
with open(_bt_cfg_name, "w") as f:
    yaml.dump(_bt_cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"Patched {_bt_cfg_name}")

# Quick sanity check on the training file
from src.data.dataset import load_pairs, print_stats
df = load_pairs(str(LOCAL_DATA / "MSLG_SPA_train.txt"))
print_stats(df, name="Training data")


## 4 - Restore checkpoints from Drive

Always run this cell before training. It is a no-op on first run and on subsequent runs it mirrors Drive's checkpoint directories into local. Completed subtasks are skipped automatically by the training cells.


In [ ]:
# 4.0 - Restore any existing checkpoints from Drive
import shutil
from pathlib import Path

LOCAL_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
LOCAL_CKPT_ROOT.mkdir(parents=True, exist_ok=True)

copied = 0
if DRIVE_CKPT.exists():
    for item in DRIVE_CKPT.iterdir():
        target = LOCAL_CKPT_ROOT / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
        copied += 1
        print(f"  restored  {item.name}")

if copied == 0:
    print("No checkpoints on Drive. Fresh training run.")
else:
    print(f"Restored {copied} item(s) from Drive.")


## 5 - Training

Pick a config (`baseline.yaml` for the reference run, `baseline_bt.yaml` for baseline + back-translated data with clean val split, or `strong.yaml` for the LoRA r=64 upgrade) and run both subtasks.

**Memory note** — `strong.yaml` has 34.6M trainable params. If you hit OOM on T4, lower `per_device_train_batch_size` in the config cell below to 4.


In [ ]:
# 5.1 - Choose config
# ============================================================
#  EDIT HERE
# ============================================================
CONFIG_NAME = "baseline_bt.yaml"  # or "baseline.yaml" / "strong.yaml"
# ============================================================

CONFIG_PATH = f"configs/{CONFIG_NAME}"
import yaml
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
print(f"Config        : {CONFIG_PATH}")
print(f"Model         : {cfg['model']['name']}")
print(f"LoRA r        : {cfg['lora']['r']}")
print(f"LoRA targets  : {cfg['lora'].get('target_modules', '[q_proj,v_proj]')}")
print(f"Epochs        : {cfg['training']['num_train_epochs']}")
print(f"Batch         : {cfg['training']['per_device_train_batch_size']}")
print(f"Label smooth  : {cfg['training'].get('label_smoothing_factor', 0.0)}")


In [ ]:
# 5.2 - Train MSLG2SPA
# Output dir from config is relative, so checkpoints land in /content/mslg-spa-2026/<output_dir>
import os
os.chdir(str(PROJECT_ROOT))

!python scripts/train.py --config {CONFIG_PATH} --subtask mslg2spa


In [ ]:
# 5.3 - Train SPA2MSLG
import os
os.chdir(str(PROJECT_ROOT))

!python scripts/train.py --config {CONFIG_PATH} --subtask spa2mslg


In [ ]:
# 5.4 - Backup all checkpoints to Drive
import shutil
from pathlib import Path

LOCAL_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
count = 0
for item in LOCAL_CKPT_ROOT.iterdir():
    target = DRIVE_CKPT / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
    count += 1
    print(f"  backed up  {item.name}")

print(f"\nBacked up {count} item(s) to {DRIVE_CKPT}")


## 6 - Submission

Generates **three submission sets** for both subtasks:
- **A** `baseline` — single-best model trained without back-translation
- **B** `baseline_bt` — single-best model trained with back-translated data
- **C** `baseline_bt_ensemble3` — top-3 checkpoint ensemble from the BT model

Run cell 6.1 first (sets params), then 6.2–6.4 in any order. Cell 6.5 is an optional val sanity check (no test labels needed). Cell 6.6 copies everything to Drive.


In [ ]:
# 6.1 - Submission parameters
# ============================================================
#  EDIT HERE
# ============================================================
TEAM_NAME     = "mslgTeam"   # your team name
N_CHECKPOINTS = 3             # top-N for ensemble
# ============================================================
import os, yaml
os.chdir(str(PROJECT_ROOT))
print(f"Team          : {TEAM_NAME}")
print(f"N checkpoints : {N_CHECKPOINTS}")


In [ ]:
# 6.2 - Submission A: baseline (no BT) — single-best model
# Outputs: {TEAM_NAME}_baseline_MSLG2SPA.txt
#          {TEAM_NAME}_baseline_SPA2MSLG.txt
import os
os.chdir(str(PROJECT_ROOT))
!python scripts/predict.py --config configs/baseline.yaml \
    --subtask mslg2spa --team {TEAM_NAME} --solution baseline
!python scripts/predict.py --config configs/baseline.yaml \
    --subtask spa2mslg --team {TEAM_NAME} --solution baseline


In [ ]:
# 6.3 - Submission B: baseline_bt (with BT) — single-best model
# Outputs: {TEAM_NAME}_baseline_bt_MSLG2SPA.txt
#          {TEAM_NAME}_baseline_bt_SPA2MSLG.txt
import os
os.chdir(str(PROJECT_ROOT))
!python scripts/predict.py --config configs/baseline_bt.yaml \
    --subtask mslg2spa --team {TEAM_NAME} --solution baseline_bt
!python scripts/predict.py --config configs/baseline_bt.yaml \
    --subtask spa2mslg --team {TEAM_NAME} --solution baseline_bt


In [ ]:
# 6.4 - Submission C: baseline_bt top-3 ensemble
# Outputs: {TEAM_NAME}_baseline_bt_ensemble3_MSLG2SPA.txt
#          {TEAM_NAME}_baseline_bt_ensemble3_SPA2MSLG.txt
import os, yaml
os.chdir(str(PROJECT_ROOT))
with open('configs/baseline_bt.yaml') as f:
    _bt = yaml.safe_load(f)
_bt_base = _bt['training']['output_dir']
print(f"Checkpoint base: {_bt_base}")

!python scripts/ensemble_predict.py --config configs/baseline_bt.yaml \
    --subtask mslg2spa \
    --checkpoint_dir {_bt_base}/mslg2spa \
    --team {TEAM_NAME} --solution baseline_bt_ensemble{N_CHECKPOINTS} \
    --n_checkpoints {N_CHECKPOINTS}
!python scripts/ensemble_predict.py --config configs/baseline_bt.yaml \
    --subtask spa2mslg \
    --checkpoint_dir {_bt_base}/spa2mslg \
    --team {TEAM_NAME} --solution baseline_bt_ensemble{N_CHECKPOINTS} \
    --n_checkpoints {N_CHECKPOINTS}


In [ ]:
# 6.5 - Val sanity check: single-best vs ensemble on real val split
# No test labels needed. Compares chrF of single-best vs top-3 ensemble.
# Run this before deciding which submission to use.
import os, yaml
os.chdir(str(PROJECT_ROOT))
with open('configs/baseline_bt.yaml') as f:
    _bt = yaml.safe_load(f)
_bt_base = _bt['training']['output_dir']

print("=== MSLG2SPA ===")
!python scripts/ensemble_predict.py --config configs/baseline_bt.yaml \
    --subtask mslg2spa \
    --checkpoint_dir {_bt_base}/mslg2spa \
    --validate --n_checkpoints {N_CHECKPOINTS}
print("=== SPA2MSLG ===")
!python scripts/ensemble_predict.py --config configs/baseline_bt.yaml \
    --subtask spa2mslg \
    --checkpoint_dir {_bt_base}/spa2mslg \
    --validate --n_checkpoints {N_CHECKPOINTS}


In [ ]:
# 6.6 - Copy all outputs to Drive + download locally
import shutil
from pathlib import Path
from google.colab import files

outputs_dir = PROJECT_ROOT / "outputs"
if not outputs_dir.exists() or not any(outputs_dir.iterdir()):
    print("No outputs to upload.")
else:
    for f in sorted(outputs_dir.glob("*.txt")):
        dest = DRIVE_SUB / f.name
        shutil.copy2(f, dest)
        print(f"  Drive: {dest.name}")
    print()
    print("Downloading all .txt files...")
    for f in sorted(outputs_dir.glob("*.txt")):
        files.download(str(f))
